In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "global_settings.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
from src.global_settings import settings


In [ ]:
import math
import json
from pathlib import Path
import warnings

import torch
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import seaborn as sns

from src.global_settings import settings
from src.sae import SparseAutoencoder
from src.activations import ActivationDataset
from src.utils.model_loader import load_sae
from src.features import load_score_data
from src.utils.font import setup_plt_font
from src.utils.metrics import calculate_significances, calculate_fqi
from src.utils.wordclouds import collect_cluster_tokens, show_wordcloud, show_all_wordclouds
from src.utils.explanation_loader import load_feature_explanation

setup_plt_font()
SEED = 42

In [ ]:
MODEL = "f3072_k8"
CHECKPOINT = f"{MODEL}_alpha"
LAYER = 21

META_DIR = Path(settings.feature_meta_dir)
CLUSTER_PATH = Path(settings.feature_cluster_path)

meta_df = pd.read_csv(settings.text_dataset_path)

scores_data = load_score_data(checkpoint=CHECKPOINT, layer=LAYER, metric_name="total_post")
scores_df = pd.DataFrame(scores_data)

sample_counts = (scores_df > 0).sum(axis=0)
sample_counts = sample_counts[sample_counts > 0]

MIN_ACTIVE_SAMPLES = 200
live_feature_indices = sample_counts[sample_counts > MIN_ACTIVE_SAMPLES].index.tolist()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sae: SparseAutoencoder = load_sae(checkpoint=CHECKPOINT, layer_idx=LAYER, return_ckpt=False)
sae.to(device)
sae.eval()

feature_df = pd.read_csv("./outputs/analysis/live_features_f3072_k8_L21.csv")

torch.manual_seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(SEED)

activation_dataset = ActivationDataset(data_dir=f"./data/post_mlp_residual_stream/layer_{LAYER}")
activation_loader = DataLoader(
    activation_dataset,
    batch_size=1,
    generator=g,
    num_workers=0,
)

In [ ]:
all_significance_scores = calculate_significances(scores_df, meta_df)
all_frequencies = (scores_df > 0).sum(axis=0)
all_frequencies.name = "frequency"

In [ ]:
with open(CLUSTER_PATH, "r") as f:
    cluster_dict = json.load(f)

cluster_fqis = {}
for cluster_id, feature_indices in cluster_dict.items():
    cluster_fqi = calculate_fqi(
        feature_indices=feature_indices,
        significance_map=all_significance_scores,
        frequency_map=all_frequencies,
    )
    cluster_fqis[cluster_id] = cluster_fqi

In [ ]:
df_fqi = pd.DataFrame.from_dict(cluster_fqis, orient="index", columns=["FQI"])
print(df_fqi.T.to_markdown())

In [ ]:
df_to_plot = df_fqi.reset_index().rename(columns={"index": "Cluster ID"})

df_to_plot['color_type'] = np.where(
    df_to_plot['FQI'] >= 0, 
    'Good (Positive)', 
    'Bad (Negative)'
)

df_to_plot = df_to_plot.sort_values(by="FQI", ascending=False)
df_to_plot["Cluster ID"] = df_to_plot["Cluster ID"].astype(str)

plt.figure(figsize=(10, 6))
color_palette = {
    'Good (Positive)': '#3498db', # (Blue)
    'Bad (Negative)': '#e74c3c'  # (Red)
}    
sns.barplot(
    x="Cluster ID",
    y="FQI",
    data=df_to_plot,
    hue='color_type',
    palette=color_palette,
    dodge=False,
)
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.title("Feature Cluster Quality (FQI) Scores by Cluster", fontsize=18, weight='bold')
plt.xlabel("Cluster ID (Sorted by FQI)", fontsize=12)
plt.ylabel("FQI Score", fontsize=12)
plt.xticks(rotation=45, fontsize=10)
plt.legend(title='Cluster Type', loc='upper right')

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
total_model_fqi = calculate_fqi(
    feature_indices=live_feature_indices,
    significance_map=all_significance_scores,
    frequency_map=all_frequencies,
)
print("-"*50)
print(f"Total Model FQI (n={len(live_feature_indices)}): {total_model_fqi:.4f}")

In [ ]:
feature_to_tokens = collect_cluster_tokens(
    activation_loader=activation_loader,
    sae=sae,
    num_samples=1000,
)

show_all_wordclouds(
    clusters=cluster_dict,
    feature_to_tokens=feature_to_tokens,
    fqi_map=cluster_fqis,
    skip_noise_cluster=False,
)

In [ ]:
best_features = list(cluster_dict["19"])

print(f"Best features in cluster 19: {best_features}")

In [ ]:
explanations = {}
for feat_idx in best_features:
    exp, des = load_feature_explanation(CHECKPOINT, LAYER, feat_idx)
    explanations[feat_idx] = {
        "explanation": exp,
        "description": des,
    }

for feat_idx, explanation in explanations.items():
    print(f"\nFeature #{feat_idx}:")
    print(explanation["description"])
    print(explanation["explanation"])

# 모델 통째로 cluster로 봤을 때 Word cloud

In [ ]:
final_cluster = {1: []}
for cluster_id, feature_indices in cluster_dict.items():
    final_cluster[1].extend(feature_indices)

In [ ]:
show_wordcloud(cluster_id=1, clusters=final_cluster, feature_to_tokens=feature_to_tokens)